In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize

plt.style.use('seaborn-v0_8-darkgrid')

RF = 0.05  # annualised risk-free rate (approx. current T-bill yield)

# Assets were selected across uncorrelated classes — equities, bonds,
# commodities, and real estate — so the efficient frontier reflects genuine
# diversification benefits rather than variance within a single sector.
tickers = ['SPY', 'AAPL', 'GLD', 'TLT', 'VNQ', 'EFA', 'XLE']
asset_labels = {
    'SPY': 'US Equities', 'AAPL': 'Tech',
    'GLD': 'Gold',        'TLT': 'US Bonds',
    'VNQ': 'Real Estate', 'EFA': 'Intl Equities',
    'XLE': 'Energy',
}
start_date = '2020-01-01'

print(f"Downloading data for: {tickers}")
data = yf.download(tickers, start=start_date, progress=False, auto_adjust=True)

if isinstance(data.columns, pd.MultiIndex):
    try:
        prices = data['Close']
    except KeyError:
        prices = data

prices = prices.dropna()
log_returns = np.log(prices / prices.shift(1)).dropna()

normalized_prices = (prices / prices.iloc[0]) * 100

plt.figure(figsize=(12, 6))
plt.plot(normalized_prices)
plt.title("Portfolio Asset Performance (Normalized to 100)")
plt.xlabel("Date")
plt.ylabel("Normalized Price")
plt.legend([f"{t} — {asset_labels[t]}" for t in tickers])
plt.show()

print("Data fetched successfully. Dimensions:", log_returns.shape)

## Asset Correlation Matrix

The correlation matrix confirms low (and in some cases negative) correlation between equities, bonds, and gold.
This is the core requirement for MPT: uncorrelated assets reduce portfolio variance without proportionally reducing return.

In [ ]:
corr = log_returns.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1,
            xticklabels=tickers, yticklabels=tickers)
plt.title('Asset Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Annualised metrics (252 trading days)
mean_returns = log_returns.mean() * 252
cov_matrix   = log_returns.cov()  * 252

print("--- Annualised Expected Returns ---")
print(mean_returns.apply(lambda x: f"{x:.2%}"))

num_portfolios = 10_000
num_assets     = len(tickers)

results     = np.zeros((3, num_portfolios))
all_weights = np.zeros((num_portfolios, num_assets))

np.random.seed(42)
print("\nSimulating 10,000 portfolios...")

for i in range(num_portfolios):
    weights        = np.random.random(num_assets)
    weights       /= np.sum(weights)
    all_weights[i] = weights

    p_return  = np.sum(mean_returns * weights)
    p_vol     = np.sqrt(weights @ cov_matrix @ weights)
    p_sharpe  = (p_return - RF) / p_vol   # risk-adjusted, uses real RF rate

    results[0, i] = p_return
    results[1, i] = p_vol
    results[2, i] = p_sharpe

print("Simulation complete.")

## Maximum Sharpe Portfolio

Monte Carlo approximates the frontier by brute-force sampling; scipy's SLSQP finds the *true* analytical optimum.
Both are plotted — the gold star should sit at or above the red star.

In [ ]:
# ── Monte Carlo approximation ──────────────────────────────────────────────
max_sharpe_idx    = np.argmax(results[2])
mc_sharpe_return  = results[0, max_sharpe_idx]
mc_sharpe_vol     = results[1, max_sharpe_idx]

# ── Analytical optimisation via SLSQP ───────────────────────────────────────
n            = len(tickers)
bounds       = tuple((0, 1) for _ in range(n))
constraints  = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
init_guess   = np.full(n, 1 / n)

def neg_sharpe(weights, mean_returns, cov_matrix, rf=RF):
    p_ret = np.dot(weights, mean_returns)
    p_vol = np.sqrt(weights @ cov_matrix @ weights)
    return -(p_ret - rf) / p_vol

result = minimize(neg_sharpe, init_guess,
                  args=(mean_returns, cov_matrix),
                  method='SLSQP', bounds=bounds, constraints=constraints)

optimal_weights = result.x
opt_return      = np.dot(optimal_weights, mean_returns)
opt_vol         = np.sqrt(optimal_weights @ cov_matrix @ optimal_weights)
opt_sharpe      = (opt_return - RF) / opt_vol

print("--- Optimal Portfolio (Max Sharpe via SLSQP) ---")
print(f"Return:       {opt_return:.2%}")
print(f"Volatility:   {opt_vol:.2%}")
print(f"Sharpe Ratio: {opt_sharpe:.2f}")
print("\nAllocation:")
for ticker, weight in zip(tickers, optimal_weights):
    print(f"  {ticker:4s} ({asset_labels[ticker]}): {weight:.2%}")

# ── Plot ─────────────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 6))
plt.scatter(results[1], results[0], c=results[2], cmap='viridis', alpha=0.5, s=10)
plt.colorbar(label='Sharpe Ratio')
plt.scatter(mc_sharpe_vol,  mc_sharpe_return, c='red',  s=200, marker='*',
            label='MC Max Sharpe (approx.)')
plt.scatter(opt_vol, opt_return, c='gold', s=300, marker='*', zorder=5,
            label='SLSQP Max Sharpe (analytical)')
plt.title('Efficient Frontier: Monte Carlo + Analytical Optimum')
plt.xlabel('Annualised Volatility (Risk)')
plt.ylabel('Annualised Return')
plt.legend()
plt.show()

## Global Minimum Variance Portfolio

The GMV portfolio minimises risk regardless of return — useful as a conservative anchor.

In [ ]:
def portfolio_vol(weights, cov_matrix):
    return np.sqrt(weights @ cov_matrix @ weights)

result_mv    = minimize(portfolio_vol, init_guess,
                        args=(cov_matrix,),
                        method='SLSQP', bounds=bounds, constraints=constraints)

min_var_weights = result_mv.x
mv_return       = np.dot(min_var_weights, mean_returns)
mv_vol          = np.sqrt(min_var_weights @ cov_matrix @ min_var_weights)

print("--- Global Minimum Variance (GMV) Portfolio ---")
print(f"Return:     {mv_return:.2%}")
print(f"Volatility: {mv_vol:.2%}")
print("\nAllocation:")
for ticker, weight in zip(tickers, min_var_weights):
    print(f"  {ticker:4s} ({asset_labels[ticker]}): {weight:.2%}")

plt.figure(figsize=(10, 6))
plt.scatter(results[1], results[0], c=results[2], cmap='viridis', alpha=0.5, s=10)
plt.colorbar(label='Sharpe Ratio')
plt.scatter(opt_vol, opt_return, c='gold', s=300, marker='*', zorder=5,
            label='Max Sharpe — Aggressive (SLSQP)')
plt.scatter(mv_vol,  mv_return,  c='blue', s=200, marker='*',
            label='Min Variance — Conservative (SLSQP)')
plt.title('Efficient Frontier: Risk vs. Reward')
plt.xlabel('Annualised Volatility (Risk)')
plt.ylabel('Annualised Return')
plt.legend()
plt.show()

## Optimal Portfolio Allocations

What would you actually buy? The table below shows the recommended weights for each strategy.

In [ ]:
weights_df = pd.DataFrame({
    'Ticker':       tickers,
    'Asset Class':  [asset_labels[t] for t in tickers],
    'Max Sharpe':   [f"{w * 100:.2f}%" for w in optimal_weights],
    'Min Variance': [f"{w * 100:.2f}%" for w in min_var_weights],
})
print(weights_df.to_string(index=False))